In [ ]:
import os
import pandas as pd
import numpy as np

def export_signedgraph_txt(df: pd.DataFrame,
                           out_path: str,
                           drop_neutral: bool = True,
                           deduplicate: str = "interactions"):
    """
    Esporta un DataFrame con colonne:
      - source, target, edge_sign  (richieste)
      - opz.: interactions, total_abs, signed_weight (usate per dedup)
    nel formato .txt letto da SignedGraph:
      # N
      u \t v \t sign   (sign ∈ {+1, -1})

    Parametri:
        df : DataFrame delle interazioni aggregate.
        out_path : percorso del file .txt di output (es. '../datasets/politisky24.txt').
        drop_neutral : se True, scarta edge_sign==0 (consigliato).
        deduplicate : come risolvere più righe per la stessa coppia (u,v) non orientata:
            - "interactions"  -> tiene la riga con interactions massimo (se presente)
            - "total_abs"     -> tiene la riga con total_abs massimo (se presente)
            - "first"         -> tiene la prima occorrenza
            - "none"          -> non deduplica (sconsigliato)
    """
    df2 = df.copy()

    # 1) Clean self-loops and neutral edges
    df2 = df2[df2["source"] != df2["target"]]
    if drop_neutral:
        df2 = df2[df2["edge_sign"] != 0]

    # 2) Normalize sign to {+1, -1}
    df2["sign"] = np.where(df2["edge_sign"] > 0, 1, -1)

    # 3) Remap nodes to 0..N-1
    nodes = pd.Index(pd.unique(df2[["source", "target"]].values.ravel("K")))
    id_map = {int(node): i for i, node in enumerate(nodes)}
    df2["u"] = df2["source"].map(id_map).astype(np.int64)
    df2["v"] = df2["target"].map(id_map).astype(np.int64)

    # 4) Canonicalize to undirected pair (min, max)
    uv_min = df2[["u", "v"]].min(axis=1)
    uv_max = df2[["u", "v"]].max(axis=1)
    df2["u_min"] = uv_min
    df2["v_max"] = uv_max

    # 5) Deduplicate per coppia non orientata
    if deduplicate != "none":
        if deduplicate == "interactions" and "interactions" in df2.columns:
            idx = df2.groupby(["u_min", "v_max"])["interactions"].idxmax()
            df2 = df2.loc[idx]
        elif deduplicate == "total_abs" and "total_abs" in df2.columns:
            idx = df2.groupby(["u_min", "v_max"])["total_abs"].idxmax()
            df2 = df2.loc[idx]
        elif deduplicate == "first":
            df2 = df2.drop_duplicates(subset=["u_min", "v_max"], keep="first")
        else:
            # fallback: first
            df2 = df2.drop_duplicates(subset=["u_min", "v_max"], keep="first")

    # 6) If you want to enforce sign consistency in case of duplicates with conflicting signs,
    #    you could do a majority vote or sum of weights here.
    #    With prior aggregation, usually no action is needed.
    out_df = df2[["u_min", "v_max", "sign"]].sort_values(["u_min", "v_max"])

    # 7) Write to file
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(f"# {len(nodes)}\n")
        for r in out_df.itertuples(index=False):
            f.write(f"{int(r.u_min)}\t{int(r.v_max)}\t{int(r.sign)}\n")

    # 8) Log sintetico
    print(f"Wrote: {out_path}")
    print(f"Nodes: {len(nodes)} | Edges (undirected): {len(out_df)} "
          f"| Pos: {int((out_df['sign']==1).sum())} | Neg: {int((out_df['sign']==-1).sum())}")


In [5]:
import os

FINAL_CSV = os.path.join(
    os.getcwd(),
    "POLITISKY24_Layers",
    "cumulative_signed_layers",
    "cumulative_signed_edges.csv"
)


df_politysky = pd.read_csv(FINAL_CSV)

OUT_TXT = os.path.abspath(os.path.join("..", "datasets", "politisky24.txt"))
export_signedgraph_txt(df_politysky, OUT_TXT, drop_neutral=True, deduplicate="interactions")


Wrote: c:\Users\jacop\OneDrive - Università della Calabria\LM Intelligenza Artificiale\2 - Secondo Anno\Analisi di Social Network e Media\datasets\politisky24.txt
Nodes: 81951 | Edges (undirected): 1566730 | Pos: 1003636 | Neg: 563094
